### Step 1: Install dependencies

<br>

First make sure you have created a virtual environment (venv) and connected to it in this notebook (top right corner)

In [ ]:
!pip install langchain_core langchain-anthropic langgraph

### Step 2: Initialize the LLM

<br>

Create a .env with:
<br>
ANTHROPIC_API_KEY=your-api-key
<br>
LANGSMITH_TRACING=true

In [ ]:
import os
import getpass

from langchain_anthropic import ChatAnthropic

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")


_set_env("ANTHROPIC_API_KEY")

llm = ChatAnthropic(model="claude-haiku-4-5")

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

system_prompt="""
You are a legal analyst that extracts entities from legal documents and does data enrichment.
You must analyze the text provided and extract the names of the entities involved for data enrichment.
"""

user_prompt="""Here is a snippet from a legal document. Retrieve the historical information for the entities involved:

This Software as a Service Agreement (this "Agreement") is entered into as of January 10,
2024 (the "EKective Date"), by and between:
CloudPlatform Solutions Inc., a Delaware corporation ("Vendor"), with its principal place of
business at 800 Cloud Drive, Seattle, WA 98101
and
Acme Legal LLP, a limited liability partnership organized under the laws of New York
("Customer"), with its principal place of business at 425 Park Avenue, New York, NY 10022
RECITALS
WHEREAS, Vendor provides cloud-based legal practice management software and related
services;
WHEREAS, Customer desires to subscribe to and use Vendor's software platform and
services; and
WHEREAS, Vendor desires to provide such services to Customer under the terms and
conditions set forth in this Agreement;
NOW, THEREFORE, in consideration of the mutual covenants and agreements hereinafter
set forth and for other good and valuable consideration, the receipt and suKiciency of
which are hereby acknowledged, the parties agree as follows:

"""

# Invoke the LLM with a list of messages
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_prompt)
]

response = llm.invoke(messages)
print(response.content)


### Step 3: Implement Data Enrichment Tool

In [ ]:
# Define a tool
def get_entity_history(entity_name: str, days_back: int = 30):
    """
    Retrieves historical information about a legal entity.
    (Mock version that returns fake data for demonstration)
    
    Args:
        entity_name: The name of the legal entity to look up
        days_back: Number of days of history to retrieve (default: 30)
    
    Returns:
        dict: JSON response containing entity history
    """
    # Return mock data for demonstration
    return {
        "entity_name": entity_name,
        "days_back": days_back,
        "history": [
            {"date": "2024-01-15", "event": "Contract signed", "value": "$50,000"},
            {"date": "2024-01-20", "event": "Service activated", "value": "$0"},
            {"date": "2024-02-01", "event": "First billing", "value": "$50,000"}
        ],
        "total_transactions": 3
    }

In [ ]:
# Augment the LLM with tools
llm_with_tools = llm.bind_tools([get_entity_history])


In [ ]:
# Invoke the LLM with tools
response = llm_with_tools.invoke(messages)

# Check if the LLM wants to call a tool
print("Response type:", type(response))
print("\nDid the LLM call a tool?", len(response.tool_calls) > 0)

if response.tool_calls:
    print("\n--- Tool Calls Requested ---")
    for tool_call in response.tool_calls:
        print(f"\nTool: {tool_call['name']}")
        print(f"Arguments: {tool_call['args']}")
else:
    print("\n--- Direct Response ---")
    print(response.content)

In [ ]:
# Execute Tools and Complete the Loop

# This shows the full tool calling pattern:
# 1. LLM requests tool calls
# 2. We execute the tool
# 3. We send results back to the LLM
# 4. LLM provides final answer

from langchain_core.messages import ToolMessage

# Step 1: Get initial response with tool calls
response = llm_with_tools.invoke(messages)

if response.tool_calls:
    print("=== LLM requested tool calls ===\n")
    
    # Step 2: Execute each tool call
    tool_messages = []
    for tool_call in response.tool_calls:
        print(f"Executing: {tool_call['name']}({tool_call['args']})")
        
        # Execute the tool function
        result = get_entity_history(**tool_call['args'])
        print(f"Result: {result}\n")
        
        # Create a ToolMessage with the result
        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call['id']
            )
        )
    
    # Step 3: Send tool results back to LLM for final answer
    print("=== Sending results back to LLM ===\n")
    final_response = llm_with_tools.invoke(messages + [response] + tool_messages)
    print("=== Final LLM Response ===")
    print(final_response.content)
else:
    print("No tool calls were made")
    print(response.content)
